# Exploration des candidats BAAC–OSM
Exécuter les cellules dans l’ordre. Modifier uniquement la configuration de la cellule suivante. Aucun fichier n'est écrit : le rapport de production reste la responsabilité de `src/report_candidates.py`. Les accidents hors de la boîte englobante des routes ont été exclus **avant** la jointure et ne figurent pas dans le fichier de candidats.


In [10]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

CANDIDATES_FILE = PROJECT_ROOT / 'outputs/maps/candidates-bretagne-2024.geoparquet'
BAAC_YEAR = 2024
DEPARTMENTS = ('22', '29', '35', '56')  # Mettre None pour ne pas filtrer par département
TOP_N = 20

assert CANDIDATES_FILE.is_file(), CANDIDATES_FILE
CARACT_FILE = PROJECT_ROOT / f'BAAC/derived/{BAAC_YEAR}/caract-{BAAC_YEAR}.geoparquet'
LIEUX_FILE = PROJECT_ROOT / f'BAAC/derived/{BAAC_YEAR}/lieux-{BAAC_YEAR}.parquet'
assert CARACT_FILE.is_file(), CARACT_FILE
assert LIEUX_FILE.is_file(), LIEUX_FILE
print('Candidats :', CANDIDATES_FILE)
print('BAAC :', BAAC_YEAR, '| Départements :', DEPARTMENTS)


Candidats : /home/tritbool/Workspace/Accidentologie_EDPM/outputs/maps/candidates-bretagne-2024.geoparquet
BAAC : 2024 | Départements : ('22', '29', '35', '56')


## 1. Charger et vérifier
Une ligne est un candidat route ou un accident sans candidat. `Num_Acc` doit être traité comme une chaîne : certains fichiers peuvent le représenter comme un flottant (`202400000031.0`). Ne pas fusionner des millésimes avant d'avoir normalisé les identifiants.


In [11]:
matches = gpd.read_parquet(CANDIDATES_FILE)
caract = gpd.read_parquet(CARACT_FILE)
lieux = pd.read_parquet(LIEUX_FILE)

required_matches = {'Num_Acc', 'geometry', 'osm_way_id', 'highway', 'distance_m', 'match_status', 'osm_source'}
required_caract = {'Num_Acc', 'dep', 'com', 'agg', 'int'}
required_lieux = {'Num_Acc', 'catr', 'voie', 'vma', 'situ'}
for label, frame, required in [('candidates', matches, required_matches), ('caract', caract, required_caract), ('lieux', lieux, required_lieux)]:
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'{label}: colonnes absentes : {sorted(missing)}')

def normalize_id(series):
    return series.astype('string').str.replace(r'\.0$', '', regex=True)

for frame in (matches, caract, lieux):
    frame['Num_Acc'] = normalize_id(frame['Num_Acc'])

if matches['Num_Acc'].isna().any() or caract['Num_Acc'].isna().any():
    raise ValueError('Identifiants Num_Acc manquants dans candidats ou caract')
if not caract['Num_Acc'].is_unique:
    raise ValueError('Num_Acc non unique dans caract : vérifier les sources')

print('Lignes candidats :', len(matches), '| accidents distincts :', matches['Num_Acc'].nunique())
print('Lignes caract :', len(caract), '| lignes lieux :', len(lieux))
print('CRS candidats :', matches.crs, '| CRS caract :', caract.crs)
display(matches.head(5))


Lignes candidats : 28265 | accidents distincts : 2603
Lignes caract : 54402 | lignes lieux : 70249
CRS candidats : {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {

,Num_Acc,geometry,osm_way_id,highway,distance_m,match_status,osm_source
0,202400000031,POINT (-1.6476 48.11337),150458149,service,68.598650,candidate,bretagne-240101.geoparquet
1,202400000031,POINT (-1.6476 48.11337),150458164,service,68.569640,candidate,bretagne-240101.geoparquet
2,202400000031,POINT (-1.6476 48.11337),396610790,secondary,41.365011,candidate,bretagne-240101.geoparquet
3,202400000031,POINT (-1.6476 48.11337),150458168,service,48.005667,candidate,bretagne-240101.geoparquet
4,202400000031,POINT (-1.6476 48.11337),319650021,service,41.080828,candidate,bretagne-240101.geoparquet


## 2. Définir le périmètre
On joint `dep` depuis BAAC : le nom de l'extrait OSM ne détermine pas le département de l'accident. Les lignes hors périmètre restent dans `matches` et sont seulement écartées de `scope`.


In [12]:
caract_meta = caract[['Num_Acc', 'dep', 'com', 'agg', 'int']].copy()
scope_all = matches.merge(caract_meta, on='Num_Acc', how='left', validate='many_to_one', indicator=True)
print('Statut jointure BAAC :')
display(scope_all['_merge'].value_counts(dropna=False))
if (scope_all['_merge'] != 'both').any():
    raise ValueError('Des candidats ne correspondent pas au fichier BAAC de l’année sélectionnée')
scope = scope_all.drop(columns='_merge')
scope['dep'] = scope['dep'].astype('string').str.zfill(2)
if DEPARTMENTS is not None:
    scope = scope.loc[scope['dep'].isin(DEPARTMENTS)].copy()

print('Accidents distincts : total fichier', matches['Num_Acc'].nunique(), '| périmètre', scope['Num_Acc'].nunique())
display(scope[['Num_Acc', 'dep', 'com', 'osm_way_id', 'highway', 'distance_m', 'match_status']].head(10))


Statut jointure BAAC :


_merge
both          28265
left_only         0
right_only        0
Name: count, dtype: int64

Accidents distincts : total fichier 2603 | périmètre 2340


,Num_Acc,dep,com,osm_way_id,highway,distance_m,match_status
0,202400000031,35,35238,150458149,service,68.598650,candidate
1,202400000031,35,35238,150458164,service,68.569640,candidate
2,202400000031,35,35238,396610790,secondary,41.365011,candidate
3,202400000031,35,35238,150458168,service,48.005667,candidate
4,202400000031,35,35238,319650021,service,41.080828,candidate
5,202400000031,35,35238,467668493,service,42.361028,candidate
6,202400000031,35,35238,151625939,residential,60.121712,candidate
7,202400000031,35,35238,4913513,secondary,2.986914,candidate
8,202400000031,35,35238,150458159,service,60.707603,candidate
9,202400000031,35,35238,467668497,service,43.221736,candidate


## 3. Couverture des candidats
Le dénominateur est le nombre d'accidents distincts dans `scope`, **pas** tous les accidents BAAC de ces départements. Un accident avec plusieurs routes est compté une seule fois.


In [13]:
ids_scope = set(scope['Num_Acc'])
ids_matched = set(scope.loc[scope['osm_way_id'].notna(), 'Num_Acc'])
ids_unmatched = ids_scope - ids_matched
n_scope = len(ids_scope)
coverage = len(ids_matched) / n_scope if n_scope else float('nan')
print(f'Accidents retenus : {n_scope:,}')
print(f'Avec candidat : {len(ids_matched):,}')
print(f'Sans candidat : {len(ids_unmatched):,}')
print(f'Couverture locale : {coverage:.2%}')
print('Contrôle :', len(ids_matched) + len(ids_unmatched), '/', n_scope)
display(scope['match_status'].value_counts(dropna=False).rename('lignes').to_frame())
if (scope['osm_way_id'].notna() & scope['distance_m'].isna()).any():
    print('ATTENTION : candidats sans distance')
if (scope['osm_way_id'].isna() & scope['Num_Acc'].isin(ids_matched)).any():
    print('ATTENTION : des accidents ont à la fois des lignes avec et sans candidat')


Accidents retenus : 2,340
Avec candidat : 2,339
Sans candidat : 1
Couverture locale : 99.96%
Contrôle : 2340 / 2340


,lignes
match_status,
candidate,27979
unmatched,1


## 4. Plusieurs candidats et voie minimale
On garde la ligne complète du candidat le plus proche. En cas d'égalité, `osm_source` puis `osm_way_id` départagent de manière déterministe. Cette attribution reste une hypothèse, pas une preuve de la voie de l'accident.


In [14]:
candidates = scope.loc[scope['osm_way_id'].notna()].copy()
candidate_counts = candidates.groupby('Num_Acc').size()
display(candidate_counts.value_counts().sort_index().rename_axis('candidats').rename('accidents').to_frame().head(TOP_N))
print('Accidents à plusieurs candidats :', int((candidate_counts > 1).sum()))
nearest = (candidates.assign(_osm_id_sort=candidates['osm_way_id'].astype('string'))
           .sort_values(['Num_Acc', 'distance_m', 'osm_source', '_osm_id_sort'], kind='stable', na_position='last')
           .drop_duplicates('Num_Acc').drop(columns='_osm_id_sort').copy())
assert nearest['Num_Acc'].is_unique
assert len(nearest) == len(ids_matched)
display(nearest[['Num_Acc', 'dep', 'osm_way_id', 'highway', 'distance_m']].head(TOP_N))


,accidents
candidats,
1,148
2,201
3,217
4,158
5,110
6,83
7,94
8,95
9,89


Accidents à plusieurs candidats : 2191


,Num_Acc,dep,osm_way_id,highway,distance_m
7,202400000031,35,4913513,secondary,2.986914
33,202400000037,29,1123072435,primary,0.410681
39,202400000038,29,112182608,trunk,0.078126
53,202400000049,35,369262913,secondary,8.793267
55,202400000053,22,534092734,trunk,0.706334
66,202400000061,35,111850247,trunk,0.182516
67,202400000070,22,148686818,tertiary,2.108640
76,202400000133,35,339766020,secondary,8.479749
83,202400000159,35,435278763,secondary,2.447044
84,202400000162,35,29419119,tertiary,1.162814


## 5. Classes de routes et distances
Distribution sur la voie la plus proche : une ligne par accident. Ce tableau ne constitue pas un taux de risque par classe de voie (pas d'exposition).


In [15]:
display(nearest['highway'].fillna('<non renseigné>').value_counts().rename_axis('highway').rename('accidents').to_frame())
display(nearest['distance_m'].describe(percentiles=[.5, .75, .9, .95, .99]).rename('distance_m').to_frame())
display(nearest.sort_values('distance_m', ascending=False)[['Num_Acc', 'dep', 'osm_way_id', 'highway', 'distance_m']].head(TOP_N))


,accidents
highway,
secondary,582
tertiary,469
primary,407
trunk,273
residential,257
unclassified,164
service,113
cycleway,43
living_street,15


,distance_m
count,2339.000000
mean,3.002250
std,5.283013
min,0.000555
50%,1.537534
75%,3.505734
90%,6.681010
95%,9.139478
99%,24.607773
max,85.158389


,Num_Acc,dep,osm_way_id,highway,distance_m
20796,202400039861,56,458933959,secondary,85.158389
20747,202400039567,56,245481968,unclassified,65.045008
14149,202400027283,56,222442114,unclassified,57.026504
22315,202400043092,29,1079817810,track,55.532920
17300,202400033302,35,412843670,trunk,55.209560
15167,202400028816,22,467494876,unclassified,52.437096
15022,202400028690,35,23811952,trunk,49.483202
20026,202400038603,35,5213645,trunk,49.239317
1505,202400002398,56,24272955,residential,45.602832
20799,202400039871,35,331839999,service,45.097112


## 6. Examiner les non-appariés
La table `lieux` peut contenir plusieurs lignes pour le même accident. La jointure est donc `one_to_many` ; `len(unmatched_details)` compte des **lignes lieux**, pas des accidents distincts. Aucun doublon n'est supprimé de `lieux`.


In [16]:
unmatched = (scope.loc[scope['Num_Acc'].isin(ids_unmatched)]
             .drop_duplicates('Num_Acc').copy())
lieux_columns = ['Num_Acc', 'catr', 'voie', 'circ', 'nbv', 'vosp', 'plan', 'larrout', 'infra', 'situ', 'vma']
unmatched_details = unmatched.merge(lieux[lieux_columns], on='Num_Acc', how='left',
                                   validate='one_to_many', indicator=True)
print('Accidents non appariés :', unmatched['Num_Acc'].nunique())
print('Lignes après jointure lieux :', len(unmatched_details))
display(unmatched_details['_merge'].value_counts(dropna=False))
display(unmatched_details[['Num_Acc', 'dep', 'com', 'catr', 'voie', 'situ', 'vma', 'geometry']].head(TOP_N))
display(unmatched_details.groupby('dep')['Num_Acc'].nunique().rename('accidents_distincts').to_frame())
display(unmatched_details.groupby('catr', dropna=False)['Num_Acc'].nunique().rename('accidents_distincts').to_frame())


Accidents non appariés : 1
Lignes après jointure lieux : 1


_merge
both          1
left_only     0
right_only    0
Name: count, dtype: int64

,Num_Acc,dep,com,catr,voie,situ,vma,geometry
0,202400039113,35,35051,1.0,NORMANDIE (PORTE DE),3,90,POINT (-1.60452 48.15172)


,accidents_distincts
dep,
35,1


,accidents_distincts
catr,
1.0,1


## 7. Audit du dénominateur BAAC
Compare les accidents du fichier de candidats à tous les accidents du BAAC dans les départements sélectionnés. Un accident absent du fichier de candidats a été exclu *avant* la jointure, ou n'avait pas de point exploitable : il ne faut pas l'appeler « sans route ». Le fichier `caract` GeoParquet peut lui-même omettre les accidents non géocodés ; ce contrôle ne couvre donc que les lignes présentes dans ce GeoParquet.


In [17]:
baac_scope = caract.copy()
baac_scope['dep'] = baac_scope['dep'].astype('string').str.zfill(2)
if DEPARTMENTS is not None:
    baac_scope = baac_scope.loc[baac_scope['dep'].isin(DEPARTMENTS)]
ids_baac = set(baac_scope['Num_Acc'])
ids_not_evaluated = ids_baac - ids_scope
print('Accidents dans le GeoParquet BAAC du périmètre :', len(ids_baac))
print('Évalués dans le fichier de candidats :', len(ids_scope))
print('Absents du fichier de candidats :', len(ids_not_evaluated))
print('Présents dans candidats mais hors BAAC périmètre :', len(ids_scope - ids_baac))
display(baac_scope.loc[baac_scope['Num_Acc'].isin(ids_not_evaluated), ['Num_Acc', 'dep', 'com', 'geometry']].head(TOP_N))


Accidents dans le GeoParquet BAAC du périmètre : 2340
Évalués dans le fichier de candidats : 2340
Absents du fichier de candidats : 0
Présents dans candidats mais hors BAAC périmètre : 0


,Num_Acc,dep,com,geometry


## 8. Préparer la comparaison globale
**Ne combine pas les non-appariés locaux par union :** un accident absent du réseau d'une région peut matcher dans une autre. Lorsqu'on dispose de tous les fichiers pertinents, calculer `ID BAAC - union(ID ayant au moins un candidat)`. Les accidents BAAC non évalués localement restent dans le dénominateur et doivent être distingués des accidents évalués mais sans route. `Num_Acc` est unique dans le millésime testé ; conserve l'année dans toute analyse multiannuelle.


In [18]:
print('ID sans candidat dans ce fichier seulement :', len(ids_unmatched))
print('ID évalués sans route dans ce fichier :', sorted(ids_unmatched)[:TOP_N])
print('ID BAAC non évalués par ce fichier :', len(ids_not_evaluated))
# Après traitement des autres régions, unionner leurs ID appariés ; ne pas conclure sur les non-appariés globaux ici.


ID sans candidat dans ce fichier seulement : 1
ID évalués sans route dans ce fichier : ['202400039113']
ID BAAC non évalués par ce fichier : 0
